# 1. Definição das Funções de Limpeza

In [0]:
from pyspark.sql import functions as F

# ---------- Funções Auxiliares (Regras de Limpeza) ----------
def _apply_name_rules(colname: str) -> str:
    """Aplica as regras do professor: Maiúsculas e substituição de siglas."""
    n = colname.upper()
    n = n.replace("CD_", "CODIGO_")
    n = n.replace("COD_", "CODIGO_")
    n = n.replace("VL_", "VALOR_")
    n = n.replace("DT_", "DATA_")
    n = n.replace("NM_", "NOME_")
    n = n.replace("DS_", "DESCRICAO_")
    n = n.replace("NR_", "NUMERO_")
    n = n.replace("_UF", "_UNIDADE_FEDERATIVA")
    return n

def _safe_drop(df, cols):
    """Remove colunas antigas da Bronze para não acumular lixo na Silver."""
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df

# ---------- Função Principal de Processamento ----------
def processar_silver(tabela: str):
    """
    Lê da Bronze, limpa colunas, adiciona metadados Silver e salva na Silver.
    """
    src_fqn = f"workspace.bronze.{tabela}"
    dest_fqn = f"workspace.silver.{tabela}"
    
    print(f"Processando: {src_fqn} -> {dest_fqn}")

    # 1. Lê a tabela da Bronze
    df = spark.read.table(src_fqn)

    # 2. Renomeia todas as colunas conforme as regras
    new_cols = [_apply_name_rules(c) for c in df.columns]
    df = df.toDF(*new_cols)

    # 3. Remove metadados da camada anterior (Bronze) E o lixo técnico
    # IMPORTANTE: Como você renomeou tudo para MAIÚSCULO no passo 2, 
    # o drop deve usar os nomes em maiúsculo também.
    colunas_para_remover = [
        "DATA_HORA_BRONZE", "FONTE_DADOS", "NOME_ARQUIVO", 
        "HASH_ATIVACAO_USUARIO", "TOKEN_SESSAO", "LOG_REPLICACAO"
    ]
    df = _safe_drop(df, colunas_para_remover)

    # 4. Adiciona novos metadados da camada Silver
    df = (df
          .withColumn("ORIGEM_BRONZE", F.lit(src_fqn))
          .withColumn("DATA_PROCESSAMENTO_SILVER", F.current_timestamp())
         )

    # 5. Salva como Managed Table na Silver
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true") 
       .saveAsTable(dest_fqn))
    
    print(f"   ✓ Tabela {tabela} finalizada na Silver.")

# 2. Executar o processamento para todas as tabelas

Função para a sua lista real de tabelas que vieram do Supabase.

In [0]:
# Lista das suas tabelas de Ouvidoria
minhas_tabelas = [
    "estado", 
    "cidade", 
    "usuario", 
    "ouvidoria", 
    "tipo_ouvidoria", 
    "servico_afetado", 
    "anexo"
]

# Roda o processo para cada uma
for t in minhas_tabelas:
    processar_silver(t)

print("\n--- TODAS AS TABELAS FORAM PARA A CAMADA SILVER! ---")

#3. Células de Validação (SQL)

Ver as tabelas na Silver:

In [0]:
%sql
SHOW TABLES IN workspace.silver;

Ver se as colunas estão em MAIÚSCULO e limpas (Exemplo: Usuário):

In [0]:
%sql
SELECT * FROM workspace.silver.usuario LIMIT 10;

Verificar os detalhes técnicos:

In [0]:
%sql
DESCRIBE EXTENDED workspace.silver.ouvidoria;